In [1]:
print("Let's Start")

Let's Start


In [2]:
import re
import string
import automac
import mlflow
import mlflow.sklearn
import dagshub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score 
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [3]:
df = pd.read_csv('/mnt/WindowsD/Artificial Intelligence/Datasets/tweets_emotions.csv' , index_col=0).drop(columns='tweet_id')
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [4]:
x=(df['sentiment']=='happiness') | (df['sentiment'] == 'sadness')
df = df[x]
df['sentiment'] =df['sentiment'].replace({'sadness' : 0 , 'happiness' : 1}) 
x_ = df['content'].apply(automac.text.transform)
y = df['sentiment']
y = pd.to_numeric(y, errors='raise').astype(int)


In [5]:
try:
    dagshub.init(repo_owner='jubito27', repo_name='mlops-mini-project', mlflow=True)
    mlflow.set_tracking_uri('https://dagshub.com/jubito27/mlops-mini-project.mlflow')
except Exception as e:
    print("Error occured as :" , e)
mlflow.set_experiment('LR with HyperParameter Tuning')

Accessing as jubito27

Initialized MLflow to track repo "jubito27/mlops-mini-project"

Repository jubito27/mlops-mini-project initialized!

2026/11/14 14:10:18 INFO mlflow.tracking.fluent: Experiment with name 'LR with HyperParameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ca4db727f42a45df9b113ad5b2c12da2', creation_time=1794645619033, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1794645619033, lifecycle_stage='active', name='LR with HyperParameter Tuning', tags={}, trace_location=None, workspace='default'>

In [9]:
bow = CountVectorizer(max_features=1000)
x_trans = bow.fit_transform(x_)
x = pd.DataFrame(x_trans.toarray())

In [10]:

x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , random_state=42)

In [6]:
param_grid = {
    'C' : [0.1 , 1, 10],
    'penalty' : ['l1' , 'l2'],
    'solver' : ['liblinear']

}

In [12]:
from sklearn.model_selection import GridSearchCV
with mlflow.start_run():
    gcv = GridSearchCV(LogisticRegression() , param_grid=param_grid , cv=5 ,scoring='f1' , n_jobs=-1)
    gcv.fit(x_train , y_train)

    for params , mean , std in zip(gcv.cv_results_['params'] , gcv.cv_results_['mean_test_score'] , gcv.cv_results_['std_test_score']):
        with mlflow.start_run(run_name=f"LR with Params {params}" , nested=True):
            model = LogisticRegression(**params)
            model.fit(x_train , y_train) 

            y_pred = model.predict(x_test)

            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)

            mlflow.log_params(params)
            mlflow.log_metric("cv_mean_f1", mean)
            mlflow.log_metric("cv_std_f1", std)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)                          # Log evaluation metrics

    best_param = gcv.best_params_
    best_score = gcv.best_score_

    mlflow.log_params(best_param)
    mlflow.log_metric('Best Score', best_score)

print("Best params : " , best_param)
print("Best Score : " , best_score)



🏃 View run LR with Params {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/3/runs/a5ce31d5d7cc437f958c1697d9f33826
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/3
🏃 View run LR with Params {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/3/runs/21a568d387674e5cb9882dd1ca30c92c
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/3
🏃 View run LR with Params {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/3/runs/e33c2d3d1d5947479ce84332ae2e0762
🧪 View experiment at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/3
🏃 View run LR with Params {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/jubito27/mlops-mini-project.mlflow/#/experiments/3/run

In [4]:
import pandas as pd

sdf = pd.read_csv('/mnt/WindowsD/Artificial Intelligence/MLOPS/projects/mlops-mini-project/reports/figures/scores.csv', index_col=0)

In [12]:
sdf['lightgbm']['accuracy']

np.float64(0.781099324975892)

In [19]:
sdf.loc[sdf.index[0]].values

array([0.78109932])

In [25]:
for i in range(sdf.shape[0]):
    print(f"{sdf.index[i]} :{round(sdf.loc[sdf.index[i]].values[0] , 5)}")

accuracy :0.7811
precision :0.78105
recall :0.78118
f1 :0.78106
auc :0.86893
